<a href="https://colab.research.google.com/github/samriddhi-m1227/EV_Analysis-CS163Capstone/blob/main/get_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Setup: basic imports and data files.
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import requests
import time

BASE_DIR = "/content/drive/MyDrive/cs 163-capstone"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Dataset 1. CalMatters data ([MAIN DATASET] EV registration (dmv), socioeconomic, zillow home data):

ev_path = os.path.join(DATA_DIR, "ev-zipcode-demographics.csv")
ev = pd.read_csv(ev_path, dtype={"ZIP": str})

print("EV dataset shape:", ev.shape)

ev.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cs 163-capstone/data/ev-zipcode-demographics.csv'

In [ ]:
# Dataset 2: ACS data - we are extracting a couple more columns to add on to the ev data.

In [ ]:
"""
The following ACS tables (the codes below) were selected to capture income inequality, housing tenure, housing type, vehicle access, and poverty status
all of which are relevant to EV adoption disparities and charging accessibility.
"""
CATS = []

CATS.append("B19083_001E")   # income inequality: Gini index
CATS.append("B25003") # Tenure (rent vs own)
CATS.append("B25024") # Housing structure (single vs multi-unit)
CATS.append("B08201") # Vehicle availability
CATS.append("B17001") # Poverty status

In [ ]:
# ACS Data Retrieval (DO NOT RUN AGAIN: Previously Executed)
"""
Additional socioeconomic variables were
retrieved programmatically from the American Community Survey
(ACS) 5-Year API at the ZIP Code Tabulation Area (ZCTA) level.

The script iterated over each ZIP code in the EV dataset and queried
selected ACS tables using the Census API. For each ZIP code, it:
    • Requested the specified ACS variables
    • Parsed the returned JSON response
    • Constructed a single row of ZIP-level features
    • Appended results to a CSV file (acs_extra_data.csv)
    • Skipped already-processed ZIP codes to allow safe resumption

Because this process requires many API calls and is subject to runtime
constraints, it was executed once and the resulting file
(acs_extra_data.csv) was saved locally and mounted to Google Drive.

In this notebook, we load the saved file directly to ensure
reproducibility, efficiency, and a clean data integration workflow.

"""
import requests
import pandas as pd
import time
import os
from google.colab import userdata

API_KEY = userdata.get("ACS_API_KEY")
YEAR = 2021

# Save these "extra ACS features" into their own file
ACS_FILE = "acs_extra_data.csv"

# Load existing ZIPs if file exists
if os.path.exists(ACS_FILE):
    existing_acs = pd.read_csv(ACS_FILE)
    done_zips = set(existing_acs["zip code tabulation area"].astype(int))
else:
    done_zips = set()

for i in range(len(ev["ZIP"])):
    zipcode = str(ev["ZIP"].iloc[i])

    if int(zipcode) in done_zips:
        continue

    print(f"{i+1}/{len(ev['ZIP'])}: {zipcode}")

    grid = []
    for cat in CATS:
        CAT_NAME = cat if "_" in cat else f"group({cat})"
        GEO_QUERY = f"zip%20code%20tabulation%20area:{zipcode}"
        url = f"https://api.census.gov/data/{YEAR}/acs/acs5?get=NAME,{CAT_NAME}&for={GEO_QUERY}&key={API_KEY}"

        resp = requests.get(url)
        data = resp.json()

        headers, result = data[0], data[1]
        grid.append(dict(zip(headers, result)))

    if not grid:
        continue

    row = (
        pd.DataFrame(grid)
        .groupby("zip code tabulation area")
        .first()
        .reset_index()
        .dropna(axis=1, how="all")
    )

    write_header = not os.path.exists(ACS_FILE)
    row.to_csv(ACS_FILE, mode="a", index=False, header=write_header)

    done_zips.add(int(zipcode))
    time.sleep(1)

SecretNotFoundError: Secret ACS_API_KEY does not exist.

In [ ]:
# Load the ACS Features

acs_extra_path = os.path.join(DATA_DIR, "acs_extra_data.csv")

acs_extra = pd.read_csv(
    acs_extra_path,
    dtype={"zip code tabulation area": str}
)

# Standardize ZIP column name
acs_extra = acs_extra.rename(columns={
    "zip code tabulation area": "ZIP"
})

acs_extra["ZIP"] = acs_extra["ZIP"].str.zfill(5)

print("ACS extra dataset shape:", acs_extra.shape)
acs_extra.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cs 163-capstone/data/acs_extra_data.csv'

In [ ]:
# NOTE:
# Although only five ACS tables (codes) were requested via the API,
# each ACS table contains multiple detailed estimate columns.

# For example:
# - B25003 (Tenure) includes total households and renter households.
# - B25024 (Housing Structure) includes totals and multiple unit-type counts.
# - B08201 (Vehicle Availability) includes breakdowns by number of vehicles.
# - B17001 (Poverty Status) includes total population and poverty categories.

# Therefore, the returned dataset contains many columns per table(code)
# Below, we derive specific normalized ZIP-level shares from these
# detailed counts (e.g., renter share, multi-unit share, poverty share).

In [ ]:
# Derive ACS Structural Variables which we need (some feature engineering)

# rename to acs for clarity
acs = acs_extra.copy()

# Drop all columns which end with 'M' because it signifies margin-of-error fields.
acs = acs.loc[:, ~acs.columns.str.endswith("M")]

# Convert numeric columns
for col in acs.columns:
    if col not in ["ZIP", "NAME", "GEO_ID"]:
        acs[col] = pd.to_numeric(acs[col], errors="coerce")


# --- Derived ZIP-level shares ---

acs["Gini"] = acs["B19083_001E"]
# Table: B19083
# B19083_001E = Gini Index
# Range: 0 to 1 (higher = more inequality)

acs["RenterShare"] = acs["B25003_003E"] / acs["B25003_001E"] # Renter households / Total occupied households

acs["SingleFamilyShare"] = ( # (1-unit detached + 1-unit attached) / Total housing units
    acs["B25024_002E"] + acs["B25024_003E"]
) / acs["B25024_001E"]

acs["MultiUnitShare"] = ( # (2+ unit structures) / Total housing units
    acs["B25024_004E"] +
    acs["B25024_005E"] +
    acs["B25024_006E"] +
    acs["B25024_007E"] +
    acs["B25024_008E"] +
    acs["B25024_009E"]
) / acs["B25024_001E"]

acs["ZeroVehicleShare"] = acs["B08201_002E"] / acs["B08201_001E"] # Households with zero vehicles / Total households

acs["PovertyShare"] = acs["B17001_002E"] / acs["B17001_001E"] # Population below poverty level / Total population


# Keep only final structural features
acs = acs[[
    "ZIP",
    "Gini",
    "RenterShare",
    "SingleFamilyShare",
    "MultiUnitShare",
    "ZeroVehicleShare",
    "PovertyShare"
]]

print("Final ACS dataset shape:", acs.shape)

Final ACS dataset shape: (1427, 7)


/tmp/ipython-input-571/307915356.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  acs["Gini"] = acs["B19083_001E"]
/tmp/ipython-input-571/307915356.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  acs["RenterShare"] = acs["B25003_003E"] / acs["B25003_001E"] # Renter households / Total occupied households
/tmp/ipython-input-571/307915356.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [ ]:
acs.head()

,ZIP,Gini,RenterShare,SingleFamilyShare,MultiUnitShare,ZeroVehicleShare,PovertyShare
0,94027,0.5047,0.115487,1.000000,0.000000,0.009735,0.033789
1,94022,0.4986,0.197026,0.846250,0.152745,0.022305,0.030119
2,94301,0.5426,0.446466,0.584333,0.415667,0.099370,0.043670
3,94028,0.4921,0.133763,0.907373,0.092627,0.038276,0.032767
4,94024,0.4602,0.123569,0.972871,0.027129,0.012444,0.024339


In [ ]:
#Merge ev (calmatters) with acs computed features

ev_acs = ev.merge(acs, on="ZIP", how="left")
print(ev_acs.shape)
ev_acs.head()

(1427, 27)


,ZIP,Diesel,Electric,Flex_Fuel,Gasoline,Gasoline_Hybrid,Hydrogen,Natural_Gas,PHEV,Propane,...,Black_perc,BachOrHigher_perc,Total_Population,Zillow_Home_Value_Index,Gini,RenterShare,SingleFamilyShare,MultiUnitShare,ZeroVehicleShare,PovertyShare
0,94027,89,723,137,4764,380,1,0,167,0,...,1.2,84.7,7213,7397088.91,0.5047,0.115487,1.000000,0.000000,0.009735,0.033789
1,94022,228,1880,255,12682,1416,8,1,502,0,...,0.2,87.6,20069,4383734.10,0.4986,0.197026,0.846250,0.152745,0.022305,0.030119
2,94301,141,1229,163,8513,1141,5,5,352,0,...,2.5,82.0,17243,3763337.05,0.5426,0.446466,0.584333,0.415667,0.099370,0.043670
3,94028,128,645,110,4451,609,1,1,188,0,...,0.0,85.2,6582,3708911.26,0.4921,0.133763,0.907373,0.092627,0.038276,0.032767
4,94024,253,1981,246,14374,1649,12,3,571,0,...,0.9,85.1,24203,3879869.05,0.4602,0.123569,0.972871,0.027129,0.012444,0.024339


In [ ]:
ev_acs[[
    "Gini",
    "RenterShare",
    "SingleFamilyShare",
    "ZeroVehicleShare",
    "PovertyShare"
]].describe()

,Gini,RenterShare,SingleFamilyShare,ZeroVehicleShare,PovertyShare
count,1.427000e+03,1408.000000,1408.000000,1408.000000,1411.000000
mean,-9.810791e+06,0.409282,0.689468,0.063227,0.129496
std,8.030439e+07,0.191526,0.204686,0.071827,0.085789
min,-6.666667e+08,0.027778,0.000000,0.000000,0.000000
25%,4.066500e-01,0.270620,0.593950,0.027887,0.071865
50%,4.384000e-01,0.384088,0.727852,0.046791,0.106374
75%,4.715500e-01,0.525759,0.834963,0.074434,0.169702
max,6.851000e-01,1.000000,1.000000,1.000000,1.000000


In [ ]:
# Ensure Gini is numeric and within valid range (0–1)
ev_acs["Gini"] = pd.to_numeric(ev_acs["Gini"], errors="coerce")

# Set invalid values outside theoretical bounds to NaN
ev_acs.loc[ev_acs["Gini"] < 0, "Gini"] = np.nan
ev_acs.loc[ev_acs["Gini"] > 1, "Gini"] = np.nan

In [ ]:
# Dataset 3. The CalEnviro data (enviromental burdens per zip)

# ======================
# Load CalEnviroScreen 4.0 Data (2021)
# ======================

cal_path = os.path.join(DATA_DIR, "calenviroscreen40resultsdatadictionary_f_2021.xlsx")

cal = pd.read_excel(cal_path)

print("CalEnviroScreen dataset shape:", cal.shape)
cal.head(5)

CalEnviroScreen dataset shape: (8035, 58)


,Census Tract,Total Population,California County,ZIP,Approximate Location,Longitude,Latitude,CES 4.0 Score,CES 4.0 Percentile,CES 4.0 Percentile Range,...,Linguistic Isolation Pctl,Poverty,Poverty Pctl,Unemployment,Unemployment Pctl,Housing Burden,Housing Burden Pctl,Pop. Char.,Pop. Char. Score,Pop. Char. Pctl
0,6019001100,2780,Fresno,93706,Fresno,-119.781696,36.709695,93.183570,100.000000,95-100% (highest scores),...,79.374746,76.0,98.919598,12.8,93.831338,30.3,91.039290,93.155109,9.663213,99.722642
1,6077000700,4680,San Joaquin,95206,Stockton,-121.287873,37.943173,86.653790,99.987393,95-100% (highest scores),...,95.533902,73.2,98.391960,19.8,99.206143,31.2,92.281369,93.165408,9.664281,99.735250
2,6037204920,2751,Los Angeles,90023,Los Angeles,-118.197497,34.017500,82.393909,99.974786,95-100% (highest scores),...,81.553661,62.6,93.391960,6.4,61.530453,20.3,63.967047,83.751814,8.687785,95.789208
3,6019000700,3664,Fresno,93706,Fresno,-119.827707,36.734535,81.327940,99.962179,95-100% (highest scores),...,78.711598,65.7,95.351759,15.7,97.345133,35.4,96.413181,94.641227,9.817371,99.886536
4,6019000200,2689,Fresno,93706,Fresno,-119.805504,36.735491,80.745476,99.949571,95-100% (highest scores),...,86.561104,72.7,98.304020,13.7,95.288912,32.7,94.157161,95.398873,9.895964,99.949571


In [ ]:
# NOTE:
# CalEnviroScreen 4.0 includes many environmental and population
# vulnerability indicators (e.g., housing burden, asthma, poverty,
# linguistic isolation, pollution exposures, etc.).

# The CES 4.0 Score is a composite index that combines:
#   - Pollution Burden indicators
#   - Population Characteristics indicators

# Because the CES score already incorporates these components,
# we retain the population-weighted CES 4.0 Score and selected
# subcomponents (Pollution Burden, Traffic) rather than keeping
# each raw indicator separately.

In [ ]:
cal.columns.to_list()

['Census Tract',
 'Total Population',
 'California County',
 'ZIP',
 'Approximate Location',
 'Longitude',
 'Latitude',
 'CES 4.0 Score',
 'CES 4.0 Percentile',
 'CES 4.0 Percentile Range',
 'Ozone',
 'Ozone Pctl',
 'PM2.5',
 'PM2.5 Pctl',
 'Diesel PM',
 'Diesel PM Pctl',
 'Drinking Water',
 'Drinking Water Pctl',
 'Lead',
 'Lead Pctl',
 'Pesticides',
 'Pesticides Pctl',
 'Tox. Release',
 'Tox. Release Pctl',
 'Traffic',
 'Traffic Pctl',
 'Cleanup Sites',
 'Cleanup Sites Pctl',
 'Groundwater Threats',
 'Groundwater Threats Pctl',
 'Haz. Waste',
 'Haz. Waste Pctl',
 'Imp. Water Bodies',
 'Imp. Water Bodies Pctl',
 'Solid Waste',
 'Solid Waste Pctl',
 'Pollution Burden',
 'Pollution Burden Score',
 'Pollution Burden Pctl',
 'Asthma',
 'Asthma Pctl',
 'Low Birth Weight',
 'Low Birth Weight Pctl',
 'Cardiovascular Disease',
 'Cardiovascular Disease Pctl',
 'Education',
 'Education Pctl',
 'Linguistic Isolation',
 'Linguistic Isolation Pctl',
 'Poverty',
 'Poverty Pctl',
 'Unemployment',
 '

In [ ]:
# only keeping these in Calenviro since the CES score sccounts for other variables :
"""
ZIP
CES 4.0 Score (population weighted)
Pollution Burden Score (population weighted)
Traffic (population weighted)
California County (first or mode)
"""

In [ ]:
# The CalEnviroScreen dataset is reported at the census tract level.
# However, our ev_acs dataset is structured at the ZIP code level.
#
# Because multiple census tracts can belong to a single ZIP code,
# we cannot directly merge tract-level CalEnviro data with ZIP-level EV data.
# Doing so would duplicate ZIP rows and bias regression results.
#
# To correctly align geographies, we aggregate CalEnviro metrics
# from tract level to ZIP level before merging.
#
# We compute population-weighted averages of:
#   - CES 4.0 Score
#   - CES 4.0 Percentile
#
# Population-weighting ensures that larger tracts contribute proportionally
# more to the ZIP’s environmental burden measure, preventing small tracts
# from distorting the ZIP-level estimate.
#
# The result is a clean, one-row-per-ZIP environmental burden dataset
# that can be safely merged with our EV and ACS dataset.
# ---------------------------------------------------------------


In [ ]:
# Convert ZIP to string
cal["ZIP"] = cal["ZIP"].astype(str).str.strip()

# Keep only 5-digit ZIP codes
cal = cal[cal["ZIP"].str.len() == 5]

# Keep only numeric ZIPs
cal= cal[cal["ZIP"].str.isdigit()]

# Convert back to int
cal["ZIP"] = cal["ZIP"].astype(int)


zip_env = (
    cal
    .groupby("ZIP")
    .apply(lambda x: pd.Series({
        # Population-weighted CES score
        "CES_Score_ZIP": (
            (x["CES 4.0 Score"] * x["Total Population"]).sum()
            / x["Total Population"].sum()
        ),

        # Population-weighted Pollution Burden score
        "PollutionBurden_ZIP": (
            (x["Pollution Burden Score"] * x["Total Population"]).sum()
            / x["Total Population"].sum()
        ),

        # population-weighted traffic exposure
        "Traffic_ZIP": (
            (x["Traffic"] * x["Total Population"]).sum()
            / x["Total Population"].sum()
        ),

        # County (take most common county if ZIP spans multiple)
        "County": x["California County"].mode()[0]
    }))
    .reset_index()
)

/tmp/ipython-input-571/2524019267.py:20: RuntimeWarning: invalid value encountered in scalar divide
  (x["CES 4.0 Score"] * x["Total Population"]).sum()
/tmp/ipython-input-571/2524019267.py:26: RuntimeWarning: invalid value encountered in scalar divide
  (x["Pollution Burden Score"] * x["Total Population"]).sum()
/tmp/ipython-input-571/2524019267.py:32: RuntimeWarning: invalid value encountered in scalar divide
  (x["Traffic"] * x["Total Population"]).sum()
/tmp/ipython-input-571/2524019267.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [ ]:
zip_env.head()

,ZIP,CES_Score_ZIP,PollutionBurden_ZIP,Traffic_ZIP,County
0,90001,64.048429,7.258360,903.864785,Los Angeles
1,90002,63.271757,6.801310,703.763390,Los Angeles
2,90003,67.557748,7.184434,2851.878761,Los Angeles
3,90004,40.968259,6.385299,1538.492132,Los Angeles
4,90005,35.408203,6.159341,1023.001263,Los Angeles


In [ ]:
# Merge ev_acs with cal enviro

# Ensure ZIP is string in both datasets
ev_acs["ZIP"] = ev_acs["ZIP"].astype(str).str.zfill(5)
zip_env["ZIP"] = zip_env["ZIP"].astype(str).str.zfill(5)

# Merge
ev_acs_env = ev_acs.merge(zip_env, on="ZIP", how="left")

print("Merged shape:", ev_acs_env.shape)
ev_acs_cal.head(5)

Merged shape: (1427, 31)


,ZIP,Diesel,Electric,Flex_Fuel,Gasoline,Gasoline_Hybrid,Hydrogen,Natural_Gas,PHEV,Propane,...,Gini,RenterShare,SingleFamilyShare,MultiUnitShare,ZeroVehicleShare,PovertyShare,CES_Score_ZIP,PollutionBurden_ZIP,Traffic_ZIP,County
0,94027,89,723,137,4764,380,1,0,167,0,...,0.5047,0.115487,1.000000,0.000000,0.009735,0.033789,4.965985,4.938998,584.975507,San Mateo
1,94022,228,1880,255,12682,1416,8,1,502,0,...,0.4986,0.197026,0.846250,0.152745,0.022305,0.030119,4.278299,4.142339,1103.700570,Santa Clara
2,94301,141,1229,163,8513,1141,5,5,352,0,...,0.5426,0.446466,0.584333,0.415667,0.099370,0.043670,7.792235,3.983527,563.050330,Santa Clara
3,94028,128,645,110,4451,609,1,1,188,0,...,0.4921,0.133763,0.907373,0.092627,0.038276,0.032767,5.139967,4.151575,714.839738,San Mateo
4,94024,253,1981,246,14374,1649,12,3,571,0,...,0.4602,0.123569,0.972871,0.027129,0.012444,0.024339,6.942833,4.068953,822.684235,Santa Clara


In [ ]:
ev_acs_env.columns.to_list()

['ZIP',
 'Diesel',
 'Electric',
 'Flex_Fuel',
 'Gasoline',
 'Gasoline_Hybrid',
 'Hydrogen',
 'Natural_Gas',
 'PHEV',
 'Propane',
 'Total_Cars',
 'Total_EV',
 'EV_perc',
 'Median_Household_Income',
 'Latino_perc',
 'White_perc',
 'Asian_perc',
 'Black_perc',
 'BachOrHigher_perc',
 'Total_Population',
 'Zillow_Home_Value_Index',
 'Gini',
 'RenterShare',
 'SingleFamilyShare',
 'MultiUnitShare',
 'ZeroVehicleShare',
 'PovertyShare',
 'CES_Score_ZIP',
 'PollutionBurden_ZIP',
 'Traffic_ZIP',
 'County']

In [ ]:
# Save EV + ACS + CalEnviro merged dataset

output_path = os.path.join(DATA_DIR, "ev_acs_cal.csv")
ev_acs_env.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/cs 163-capstone/data/ev_acs_cal.csv


In [ ]:
# Dataset 4: nrel ev charger data
import os
import requests
import pandas as pd
import numpy as np
import io

from google.colab import userdata
NREL_API_KEY = userdata.get("NREL_API")

# Pull CA EV Charger Data

url = "https://developer.nrel.gov/api/alt-fuel-stations/v1.csv"

params = {
    "api_key": NREL_API_KEY,
    "fuel_type": "ELEC",
    "state": "CA",
    "status": "E",        # Existing stations
    "access": "public",   # Public access only
    "limit": "all"
}

response = requests.get(url, params=params)
response.raise_for_status()

stations = pd.read_csv(io.StringIO(response.text), low_memory=False)

print("Stations shape:", stations.shape)
print("Columns:", stations.columns[:20])

Stations shape: (18994, 75)
Columns: Index(['Fuel Type Code', 'Station Name', 'Street Address',
       'Intersection Directions', 'City', 'State', 'ZIP', 'Plus4',
       'Station Phone', 'Status Code', 'Expected Date',
       'Groups With Access Code', 'Access Days Time', 'Cards Accepted',
       'BD Blends', 'NG Fill Type Code', 'NG PSI', 'EV Level1 EVSE Num',
       'EV Level2 EVSE Num', 'EV DC Fast Count'],
      dtype='object')


In [ ]:
stations.describe()

,ZIP,Plus4,Expected Date,BD Blends,NG Fill Type Code,NG PSI,EV Level1 EVSE Num,EV Level2 EVSE Num,EV DC Fast Count,Latitude,...,CNG Fill Type Code,CNG PSI,CNG Vehicle Class,LNG Vehicle Class,RD Blends,RD Blends (French),RD Blended with Biodiesel,RD Maximum Biodiesel Level,CNG Station Sells Renewable Natural Gas,LNG Station Sells Renewable Natural Gas
count,18992.000000,0.0,0.0,0.0,0.0,0.0,39.000000,16617.000000,2633.000000,18994.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,93053.457245,NaN,NaN,NaN,NaN,NaN,7.205128,2.734790,6.601215,35.666205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2105.104464,NaN,NaN,NaN,NaN,NaN,11.019551,4.651234,8.377748,2.078072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,9048.000000,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,32.543738,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,91773.000000,NaN,NaN,NaN,NaN,NaN,1.000000,2.000000,1.000000,33.911220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,93014.000000,NaN,NaN,NaN,NaN,NaN,2.000000,2.000000,4.000000,34.419160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,94558.000000,NaN,NaN,NaN,NaN,NaN,9.500000,2.000000,8.000000,37.589899,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,99067.000000,NaN,NaN,NaN,NaN,NaN,51.000000,117.000000,120.000000,41.956729,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
stations.columns.to_list()

['Fuel Type Code',
 'Station Name',
 'Street Address',
 'Intersection Directions',
 'City',
 'State',
 'ZIP',
 'Plus4',
 'Station Phone',
 'Status Code',
 'Expected Date',
 'Groups With Access Code',
 'Access Days Time',
 'Cards Accepted',
 'BD Blends',
 'NG Fill Type Code',
 'NG PSI',
 'EV Level1 EVSE Num',
 'EV Level2 EVSE Num',
 'EV DC Fast Count',
 'EV Other Info',
 'EV Network',
 'EV Network Web',
 'Geocode Status',
 'Latitude',
 'Longitude',
 'Date Last Confirmed',
 'ID',
 'Updated At',
 'Owner Type Code',
 'Federal Agency ID',
 'Federal Agency Name',
 'Open Date',
 'Hydrogen Status Link',
 'NG Vehicle Class',
 'LPG Primary',
 'E85 Blender Pump',
 'EV Connector Types',
 'Country',
 'Intersection Directions (French)',
 'Access Days Time (French)',
 'BD Blends (French)',
 'Groups With Access Code (French)',
 'Hydrogen Is Retail',
 'Access Code',
 'Access Detail Code',
 'Federal Agency Code',
 'Facility Type',
 'CNG Dispenser Num',
 'CNG On-Site Renewable Source',
 'CNG Total Compre

In [ ]:
def zip5(series):
    return series.astype(str).str.extract(r"(\d{5})")[0]

# Filter to the relevant EV/public/existing/CA rows (safety check)
infra = stations.loc[
    (stations["Fuel Type Code"] == "ELEC") &
    (stations["Status Code"] == "E") &
    (stations["State"] == "CA") &
    (stations["Groups With Access Code"].str.contains("Public", case=False, na=False))
].copy()

# Time align to 2021
infra["Open Date"] = pd.to_datetime(infra["Open Date"], errors="coerce")
infra = infra[(infra["Open Date"].isna()) | (infra["Open Date"] <= "2021-12-31")]

# --- Clean ZIP ---
infra["ZIP_clean"] = zip5(infra["ZIP"])
infra = infra.dropna(subset=["ZIP_clean"])

# --- Make port counts numeric ---
for col in ["EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count"]:
    infra[col] = pd.to_numeric(infra[col], errors="coerce").fillna(0)

# --- Total ports at each station record ---
infra["Total_Ports"] = (
    infra["EV Level1 EVSE Num"] +
    infra["EV Level2 EVSE Num"] +
    infra["EV DC Fast Count"]
)

# --- Aggregate to ZIP (this is what makes merge logical) ---
infra_by_zip = (
    infra.groupby("ZIP_clean", as_index=False)
    .agg(
        Num_Stations=("ID", "nunique"),
        Total_Ports=("Total_Ports", "sum"),
        L2_Ports=("EV Level2 EVSE Num", "sum"),
        DC_Fast_Ports=("EV DC Fast Count", "sum"),
    )
)

print(infra_by_zip.shape)
infra_by_zip.head()

(1011, 5)


,ZIP_clean,Num_Stations,Total_Ports,L2_Ports,DC_Fast_Ports
0,90002,1,1.0,1.0,0.0
1,90004,2,23.0,23.0,0.0
2,90005,11,55.0,55.0,0.0
3,90006,3,20.0,20.0,0.0
4,90007,6,46.0,46.0,0.0


In [ ]:
infra_by_zip.describe()

,Num_Stations,Total_Ports,L2_Ports,DC_Fast_Ports
count,1011.000000,1011.000000,1011.000000,1011.000000
mean,8.936696,27.052423,20.640950,6.171118
std,16.971670,44.345942,40.635529,10.832176
min,1.000000,1.000000,0.000000,0.000000
25%,2.000000,4.000000,2.000000,0.000000
50%,4.000000,12.000000,8.000000,1.000000
75%,9.000000,30.000000,21.000000,8.000000
max,231.000000,518.000000,494.000000,99.000000


In [ ]:
"""
Final charger variables:

ZIP_clean:
    5-digit ZIP code used as the merge key.

Num_Stations:
    Number of unique public, operational EV charging stations
    located within the ZIP code.

Total_Ports:
    Total number of EV charging ports across all stations in the ZIP.
    (Sum of Level 1, Level 2, and DC Fast ports.)

L2_Ports:
    Total number of Level 2 charging ports in the ZIP.
    Level 2 chargers are standard public chargers (typically 240V).

DC_Fast_Ports:
    Total number of DC Fast charging ports in the ZIP.
    These provide rapid charging and are critical for long-distance travel.
"""

In [ ]:
# Merge Infrastructure (NREL) into EV+ACS+CalEnviro

# Ensure ZIP is a consistent 5-digit string key
ev_acs_cal["ZIP"] = ev_acs_cal["ZIP"].astype(str).str.zfill(5)

infra_merge = infra_by_zip.rename(columns={"ZIP_clean": "ZIP"}).copy()
infra_merge["ZIP"] = infra_merge["ZIP"].astype(str).str.zfill(5)

# Merge (left join keeps all ZIPs from ev_acs_cal)
merged = ev_acs_cal.merge(infra_merge, how="left", on="ZIP")

# ZIPs with no public chargers get 0s
infra_cols = ["Num_Stations", "Total_Ports", "L2_Ports", "DC_Fast_Ports"]
merged[infra_cols] = merged[infra_cols].fillna(0)

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (1427, 35)


,ZIP,Diesel,Electric,Flex_Fuel,Gasoline,Gasoline_Hybrid,Hydrogen,Natural_Gas,PHEV,Propane,...,ZeroVehicleShare,PovertyShare,CES_Score_ZIP,PollutionBurden_ZIP,Traffic_ZIP,County,Num_Stations,Total_Ports,L2_Ports,DC_Fast_Ports
0,94027,89,723,137,4764,380,1,0,167,0,...,0.009735,0.033789,4.965985,4.938998,584.975507,San Mateo,1.0,2.0,2.0,0.0
1,94022,228,1880,255,12682,1416,8,1,502,0,...,0.022305,0.030119,4.278299,4.142339,1103.700570,Santa Clara,17.0,153.0,133.0,20.0
2,94301,141,1229,163,8513,1141,5,5,352,0,...,0.099370,0.043670,7.792235,3.983527,563.050330,Santa Clara,15.0,65.0,64.0,1.0
3,94028,128,645,110,4451,609,1,1,188,0,...,0.038276,0.032767,5.139967,4.151575,714.839738,San Mateo,2.0,4.0,4.0,0.0
4,94024,253,1981,246,14374,1649,12,3,571,0,...,0.012444,0.024339,6.942833,4.068953,822.684235,Santa Clara,2.0,30.0,26.0,4.0


In [ ]:
merged.shape

(1427, 35)

In [ ]:
merged.columns.to_list()

['ZIP',
 'Diesel',
 'Electric',
 'Flex_Fuel',
 'Gasoline',
 'Gasoline_Hybrid',
 'Hydrogen',
 'Natural_Gas',
 'PHEV',
 'Propane',
 'Total_Cars',
 'Total_EV',
 'EV_perc',
 'Median_Household_Income',
 'Latino_perc',
 'White_perc',
 'Asian_perc',
 'Black_perc',
 'BachOrHigher_perc',
 'Total_Population',
 'Zillow_Home_Value_Index',
 'Gini',
 'RenterShare',
 'SingleFamilyShare',
 'MultiUnitShare',
 'ZeroVehicleShare',
 'PovertyShare',
 'CES_Score_ZIP',
 'PollutionBurden_ZIP',
 'Traffic_ZIP',
 'County',
 'Num_Stations',
 'Total_Ports',
 'L2_Ports',
 'DC_Fast_Ports']

In [ ]:
# SAVE FINAL DATASET
final_path = os.path.join(DATA_DIR, "final.csv")

merged.to_csv(final_path, index=False)

print("Final dataset saved to:", final_path)

Final dataset saved to: /content/drive/MyDrive/cs 163-capstone/data/final.csv


In [ ]:
# Summary: Data Integration Pipeline
"""
This notebook constructs a unified ZIP-level dataset for analyzing
EV adoption, socioeconomic characteristics, environmental burden,
and charging infrastructure in California.

Data sources integrated:
    • EV registration data (CalMatters)
    • American Community Survey (ACS) 5-year estimates
    • CalEnviroScreen 4.0 environmental indicators
    • NREL Alternative Fuel Stations (EV charging infrastructure)

Key processing steps:
    1. Loaded and standardized all datasets using ZIP as the merge key.
    2. Retrieved ACS structural variables (Gini, renter share, housing
       structure, vehicle access, poverty) via API (executed once and saved).
    3. Derived normalized ZIP-level shares from ACS tables.
    4. Aggregated CalEnviroScreen tract-level indicators to ZIP level
       (population-weighted) and retained composite and selected sub-scores.
    5. Filtered and aggregated EV charging stations to ZIP level,
       computing number of stations and total ports.
    6. Merged all components into a single final dataset.
    7. Saved the integrated dataset as final_df.csv in Google Drive.

This notebook focuses strictly on data acquisition, cleaning, and
integration. Exploratory data analysis (EDA), visualizations,
correlation analysis, and modeling will be conducted in a separate
notebook to maintain clarity and modular workflow structure.
"""